# COMP30770: Section 2 Traditional SQL Prototype
## Cyclistic Bike Share (single-threaded baseline)

Section 2 requires a non-parallel SQL prototype. This notebook runs **SQLite** through **SQLAlchemy** in one Python process and records timings as a baseline for MapReduce in Section 3.

Input: **`trips_prototype.csv`** in this folder. Output: **`cyclistic.db`** beside this notebook.

### Steps
| # | Step | Purpose |
|---|------|---------|
| 1 | Hardware specs | Justify Volume relative to hardware |
| 2 | Open SQLite database | Environment setup |
| 3 | Load local CSV | Ingest prototype trips |
| 4 | Row count & table size | Volume baseline |
| 5 | Data quality check | Missing values scan |
| 6 | Rides by member type | Aggregation query |
| 7 | Rides by bike type | Aggregation query |
| 8 | Top 10 start stations | GROUP BY + ORDER BY |
| 9 | Popular routes | MapReduce candidate 1 (heaviest query) |
| 10 | Busiest stations (combined traffic) | MapReduce candidate 2 |
| 11 | Average ride duration by member type | MapReduce candidate 3 |


---
## Step 1: Record Hardware Specs
Report volume together with hardware: record CPU, RAM, and disk so runtimes can be interpreted in context.


In [1]:
import platform
import shutil
import sqlite3
import sys

import psutil

print("===== CPU =====")
print(f"Processor: {platform.processor() or 'n/a'}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical CPUs: {psutil.cpu_count(logical=True)}")

print("\n===== RAM =====")
vm = psutil.virtual_memory()
print(f"Total: {vm.total / (1024**3):.2f} GB")
print(f"Available: {vm.available / (1024**3):.2f} GB")

print("\n===== Disk (cwd root) =====")
usage = shutil.disk_usage(".")
print(f"Total: {usage.total / (1024**3):.2f} GB")
print(f"Free: {usage.free / (1024**3):.2f} GB")

print("\n===== Python =====")
print(sys.version)

print("\n===== SQLite =====")
print(f"sqlite3 module: {sqlite3.sqlite_version}")


===== CPU =====
Processor: Intel64 Family 6 Model 126 Stepping 5, GenuineIntel
Physical cores: 2
Logical CPUs: 4

===== RAM =====
Total: 7.77 GB
Available: 1.19 GB

===== Disk (cwd root) =====
Total: 236.86 GB
Free: 6.72 GB

===== Python =====
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

===== SQLite =====
sqlite3 module: 3.49.1


---
## Step 2: Open SQLite database
The next cell creates **`cyclistic.db`** next to this notebook (or under `Prototype/` if the kernel cwd is the repo root), recreates the `trips` table, and opens a SQLAlchemy engine for all queries. Uses local SQLite only (no server process).


In [2]:
import os
import time
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

# Path to trips_prototype.csv (this folder or repo Prototype/)
def _notebook_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


NOTEBOOK_DIR = _notebook_dir()
CSV_PATH = NOTEBOOK_DIR / "trips_prototype.csv"
if not CSV_PATH.is_file():
    alt = Path.cwd() / "Prototype" / "trips_prototype.csv"
    if alt.is_file():
        CSV_PATH = alt
        NOTEBOOK_DIR = CSV_PATH.parent
    else:
        raise FileNotFoundError(
            "trips_prototype.csv not found. Run Jupyter with cwd = Prototype, or place the CSV next to this notebook. Tried: "
            f"{NOTEBOOK_DIR / 'trips_prototype.csv'} and {alt}"
        )

DB_PATH = NOTEBOOK_DIR / "cyclistic.db"
print(f"Using CSV: {CSV_PATH.resolve()}")
print(f"SQLite DB: {DB_PATH.resolve()}")

# Fresh database file
if DB_PATH.exists():
    DB_PATH.unlink()

_url = "sqlite:///" + DB_PATH.resolve().as_posix()
engine = create_engine(_url, pool_pre_ping=True)

DDL = """
CREATE TABLE trips (
    ride_id            TEXT PRIMARY KEY,
    rideable_type      TEXT,
    started_at         TEXT,
    ended_at           TEXT,
    start_station_name TEXT,
    start_station_id   TEXT,
    end_station_name   TEXT,
    end_station_id     TEXT,
    start_lat          REAL,
    start_lng          REAL,
    end_lat            REAL,
    end_lng            REAL,
    member_casual      TEXT
);
"""

with engine.begin() as e:
    e.execute(text("DROP TABLE IF EXISTS trips"))
    e.execute(text(DDL))

print("Database and empty trips table ready.")

# Ride duration in minutes (SQLite; MySQL uses TIMESTAMPDIFF(MINUTE, started_at, ended_at))
DUR_MIN = "(julianday(ended_at) - julianday(started_at)) * 24 * 60"


def run_sql(sql: str, *, label: str | None = None):
    """Run SQL, return (DataFrame, seconds)."""
    if label:
        print(label)
    t0 = time.perf_counter()
    df = pd.read_sql(text(sql), engine)
    elapsed = time.perf_counter() - t0
    print(f"Elapsed: {elapsed:.4f}s")
    return df, elapsed


Using CSV: C:\Users\imonl\BigData\Cyclistic_Data\Prototype\trips_prototype.csv
SQLite DB: C:\Users\imonl\BigData\Cyclistic_Data\Prototype\cyclistic.db
Database and empty trips table ready.


---
## Step 3: Load Data from `trips_prototype.csv`
Read the CSV with **pandas** and load it with **`DataFrame.to_sql`**.


In [3]:
df = pd.read_csv(
    CSV_PATH,
    parse_dates=["started_at", "ended_at"],
    dtype={"ride_id": str},
)

print(f"Rows in CSV: {len(df):,}")
print(df.dtypes)

t_load = time.perf_counter()
df.to_sql("trips", engine, if_exists="append", index=False, chunksize=8000)
load_s = time.perf_counter() - t_load
print(f"Load into SQLite: {load_s:.4f}s")


Rows in CSV: 148,045
ride_id                          str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
dtype: object
Load into SQLite: 5.0804s


---
## Step 4: Row Count and Table Size
Row count and database size alongside Step 1 hardware. Size comes from the **`.db` file** and **`PRAGMA` page** statistics.


In [4]:
df_rows, t1 = run_sql(
    "SELECT COUNT(*) AS total_rows FROM trips;",
    label="===== Total rows =====",
)
display(df_rows)

print("===== Database file size =====")
t0 = time.perf_counter()
sz = DB_PATH.stat().st_size / (1024**2)
with engine.connect() as conn:
    page_count = conn.execute(text("PRAGMA page_count")).scalar()
    page_size = conn.execute(text("PRAGMA page_size")).scalar()
pragma_mb = (page_count * page_size) / (1024**2)
t2 = time.perf_counter() - t0
print(f"Elapsed: {t2:.4f}s")
df_size = pd.DataFrame(
    [
        {
            "table_name": "trips",
            "sqlite_db_file_mb": round(sz, 2),
            "pragma_approx_mb": round(pragma_mb, 2),
        }
    ]
)
display(df_size)


===== Total rows =====
Elapsed: 0.0134s


,total_rows
0,148045


===== Database file size =====
Elapsed: 0.0024s


,table_name,sqlite_db_file_mb,pragma_approx_mb
0,trips,32.16,32.16


---
## Step 5: Data Quality Check
Full-table scan for missing or invalid values.


In [5]:
df_q, t_q = run_sql(
    f"""
SELECT
    SUM(CASE WHEN ride_id IS NULL OR ride_id = ''                        THEN 1 ELSE 0 END) AS missing_ride_ids,
    SUM(CASE WHEN start_station_name IS NULL OR start_station_name = ''  THEN 1 ELSE 0 END) AS missing_start_stations,
    SUM(CASE WHEN end_station_name IS NULL OR end_station_name = ''      THEN 1 ELSE 0 END) AS missing_end_stations,
    SUM(CASE WHEN end_lat IS NULL OR end_lng IS NULL                     THEN 1 ELSE 0 END) AS missing_end_coords,
    SUM(CASE WHEN member_casual IS NULL OR member_casual = ''            THEN 1 ELSE 0 END) AS missing_member_type,
    SUM(CASE WHEN ({DUR_MIN}) <= 0 THEN 1 ELSE 0 END) AS invalid_durations
FROM trips;
    """
,
    label="===== Data quality =====",
)
display(df_q)


===== Data quality =====
Elapsed: 0.1468s


,missing_ride_ids,missing_start_stations,missing_end_stations,missing_end_coords,missing_member_type,invalid_durations
0,0,16102,17453,165,0,106


---
## Step 6: Rides by Member Type


In [6]:
df6, t6 = run_sql(
    """
SELECT
    member_casual,
    COUNT(*) AS total_rides
FROM trips
GROUP BY member_casual
ORDER BY total_rides DESC;
    """
,
    label="===== Rides by member type =====",
)
display(df6)


===== Rides by member type =====
Elapsed: 0.0976s


,member_casual,total_rides
0,member,86000
1,casual,62045


---
## Step 7: Rides by Bike Type


In [7]:
df7, t7 = run_sql(
    """
SELECT
    rideable_type,
    COUNT(*) AS total_rides
FROM trips
GROUP BY rideable_type
ORDER BY total_rides DESC;
    """
,
    label="===== Rides by bike type =====",
)
display(df7)


===== Rides by bike type =====
Elapsed: 0.1083s


,rideable_type,total_rides
0,classic_bike,59391
1,electric_bike,54176
2,docked_bike,34478


---
## Step 8: Top 10 Start Stations


In [8]:
df8, t8 = run_sql(
    """
SELECT
    start_station_name,
    COUNT(*) AS total_rides
FROM trips
WHERE start_station_name IS NOT NULL
  AND start_station_name != ''
GROUP BY start_station_name
ORDER BY total_rides DESC
LIMIT 10;
    """
,
    label="===== Top 10 start stations =====",
)
display(df8)


===== Top 10 start stations =====
Elapsed: 0.1481s


,start_station_name,total_rides
0,Streeter Dr & Grand Ave,1916
1,Michigan Ave & Oak St,1151
2,Clark St & Elm St,1107
3,Wells St & Concord Ln,1063
4,Millennium Park,1047
5,Theater on the Lake,974
6,Wells St & Elm St,924
7,Indiana Ave & Roosevelt Rd,905
8,Clark St & Lincoln Ave,894
9,Kingsbury St & Kinzie St,870


---
## Step 9: Most popular routes (MapReduce candidate 1)


In [9]:
df9, t9 = run_sql(
    """
SELECT
    start_station_name,
    end_station_name,
    COUNT(*) AS route_count
FROM trips
WHERE start_station_name IS NOT NULL AND start_station_name != ''
  AND end_station_name   IS NOT NULL AND end_station_name   != ''
GROUP BY start_station_name, end_station_name
ORDER BY route_count DESC
LIMIT 5;
    """
,
    label="===== Popular routes (top 5) =====",
)
display(df9)


===== Popular routes (top 5) =====
Elapsed: 0.2185s


,start_station_name,end_station_name,route_count
0,Streeter Dr & Grand Ave,Streeter Dr & Grand Ave,322
1,Millennium Park,Millennium Park,163
2,Michigan Ave & Oak St,Michigan Ave & Oak St,161
3,Ellis Ave & 60th St,University Ave & 57th St,136
4,Ellis Ave & 60th St,Ellis Ave & 55th St,130


---
## Step 10: Busiest stations, combined traffic (MapReduce candidate 2)


In [10]:
df10, t10 = run_sql(
    """
SELECT
    station_name,
    SUM(activity_count) AS total_activity
FROM (
    SELECT start_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips
    WHERE start_station_name IS NOT NULL AND start_station_name != ''
    GROUP BY start_station_name

    UNION ALL

    SELECT end_station_name AS station_name, COUNT(*) AS activity_count
    FROM trips
    WHERE end_station_name IS NOT NULL AND end_station_name != ''
    GROUP BY end_station_name
) AS combined_traffic
GROUP BY station_name
ORDER BY total_activity DESC
LIMIT 5;
    """
,
    label="===== Busiest stations =====",
)
display(df10)


===== Busiest stations =====
Elapsed: 0.3957s


,station_name,total_activity
0,Streeter Dr & Grand Ave,3868
1,Michigan Ave & Oak St,2265
2,Clark St & Elm St,2190
3,Wells St & Concord Ln,2160
4,Millennium Park,2109


---
## Step 11: Average ride duration by member type (MapReduce candidate 3)


In [11]:
df11, t11 = run_sql(
    f"""
SELECT
    member_casual,
    ROUND(AVG({DUR_MIN}), 2) AS avg_duration_minutes,
    ROUND(MIN({DUR_MIN}), 2) AS min_duration_minutes,
    ROUND(MAX({DUR_MIN}), 2) AS max_duration_minutes
FROM trips
WHERE ({DUR_MIN}) > 0
GROUP BY member_casual;
    """
,
    label="===== Average ride duration =====",
)
display(df11)


===== Average ride duration =====
Elapsed: 0.2836s


,member_casual,avg_duration_minutes,min_duration_minutes,max_duration_minutes
0,casual,32.32,0.02,20679.02
1,member,13.70,0.02,1499.93


---
## Summary: Performance Baseline

Record the elapsed times printed above and fill in the table below for your report.

| Step | Operation | Time (s) | MapReduce Candidate? |
|------|-----------|----------|----------------------|
| 4 | Row count | | No |
| 5 | Data quality scan | | No |
| 6 | Rides by member type | | No |
| 7 | Rides by bike type | | No |
| 8 | Top 10 start stations | | No |
| 9 | Popular routes | | Yes (candidate 1) |
| 10 | Busiest stations | | Yes (candidate 2) |
| 11 | Average ride duration | | Yes (candidate 3) |

Steps 9, 10 and 11 are carried forward to Section 3 for MapReduce optimisation. Step 9 is expected to show the greatest speedup as it involves the largest intermediate result set (all unique route pairs).
